# Environment

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

import matplotlib.pyplot as plt

import os

In [ ]:
wd = os.getcwd()
print(f"El directorio de trabajo actual es: {wd}")

source_tables_path = '/home/alvaro.ritore.ext/projects/trIAje'
os.chdir(source_tables_path)

wd = os.getcwd()
print(f"El directorio de trabajo actual es: {wd}")

# Data Loading

## EDATOSCASO

In [ ]:
df_edatoscaso = pq.read_table('data/20_24_CCR_HIST_EDATOS(1)_anonimizado.parquet').to_pandas()

df_edatoscaso.info()

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# La anonimización convirtió todos los valores nan en un hash concreto. Se pasan a nan de vuelta
id_tarjeta_deseado = '81SNNM0DHR'  # Asegúrate de que el ID sea una cadena si así está en tu DataFrame
# Utiliza .loc para seleccionar las filas donde la condición se cumple y asigna NaN
df_edatoscaso.loc[df_edatoscaso['IDTARJETASANITARIA'] == id_tarjeta_deseado, 'IDTARJETASANITARIA'] = np.nan
# Para verificar el cambio, puedes contar los NaN en la columna
num_nan = df_edatoscaso['IDTARJETASANITARIA'].isnull().sum()
print(f"\nNúmero de NaN en IDTARJETASANITARIA después de la conversión: {num_nan}")

print("Number of unique patient id is: ", df_edatoscaso['IDTARJETASANITARIA'].nunique())

In [ ]:
df_edatoscaso.head(100)

## Llamadas

In [ ]:
df_llamadas_2020 = pq.read_table('data/2020_CCR_HIST_OtrasTablas.parquet').to_pandas()
df_llamadas_2021 = pq.read_table('data/2021_CCR_HIST_OtrasTablas.parquet').to_pandas()
df_llamadas_2022 = pq.read_table('data/2022_CCR_HIST_OtrasTablas.parquet').to_pandas()
df_llamadas_2023 = pq.read_table('data/2023_CCR_HIST_OtrasTablas.parquet').to_pandas()
df_llamadas_2024 = pq.read_table('data/2024_CCR_HIST_OtrasTablas.parquet').to_pandas()

# Concatenate all tables
lista_dfs = [df_llamadas_2020, df_llamadas_2021, df_llamadas_2022, df_llamadas_2023, df_llamadas_2024]
df_llamadas = pd.concat(lista_dfs, ignore_index=True)
print(f"The size of df_llamadas is: {df_llamadas.shape}")

df_llamadas.info()

# Variables to drop
df_llamadas_var_drop = ['FECHADEMANDA', 'COORDX', 'COORDY', 'DIRPACIENTEIDPROVINCIA', 
                        'DIRPACIENTEIDMUNICIPIO', 'DIRPACIENTEIDLOCALIDAD', 'DIRPACIENTELOCALIDAD']
df_llamadas_reduced = df_llamadas.drop(columns = df_llamadas_var_drop)




In [ ]:
df_llamadas_reduced.head()

## ASISTENCIA

In [ ]:
df_asistencia = pd.read_csv('data/dbo.ASISTENCIA.csv', sep = ';')
df_asistencia.info()
df_asistencia.head()


In [ ]:
df_demanda = pd.read_csv('data/dbo.DEMANDA.csv', sep = ';')
df_demanda.info()
df_demanda.head()

## Variables to drop

In [ ]:
# IDTIPODEMANDA: It doesn't provide any information of interest. All values are equal to 'A'
print(df_edatoscaso['IDTIPODEMANDA'].value_counts())

# IDTIPODEMANDA1 and IDTIPODEMANDA2: additional information outside of the scope
# HORAENTRADACONTACTO: information on date and time are contained in other columns

# Columns to drop
columns_to_drop = ['IDTIPODEMANDA', 'IDTIPODEMANDA2', 'IDTIPODEMANDA3', 'HORAENTRADACONTACTO']

df_edatoscaso_reduced = df_edatoscaso.drop(columns = columns_to_drop)
df_edatoscaso_reduced.info()

In [ ]:
df_edatoscaso_reduced.head(100)

# Data Cleaning/Filtering

In [ ]:
# Before excluding
print(f"The number of rows before applying any exclusion criteria is: {df_edatoscaso_reduced.shape[0]:,}")

df_edt_cleaned = df_edatoscaso_reduced.copy()

## Tipo Demanda

In [ ]:
id_demanda_triaje = [36, 58, 16, 23, 54]
df_edt_cleaned = df_edt_cleaned[df_edt_cleaned['IDTIPODEMANDA1'].isin(id_demanda_triaje)]

print(df_edt_cleaned['IDTIPODEMANDA1'].value_counts())

print(f"\nAfter excluding by reason for the call (IDTIPODEMANDA1), the number of rows is: {len(df_edt_cleaned):,}")

## EDAD/FNACIMIENTO

In [ ]:
df_edt_cleaned['EDAD'].hist(bins=20, range=(0, 100), color='skyblue', edgecolor='black')
plt.title('Histograma de Edad')
plt.xlabel('Edad')
plt.ylabel('Frecuencia')
plt.grid(True, linestyle='--', alpha=0.6) # Añade una cuadrícula tenue
plt.show()

df_edt_cleaned.boxplot(column=['EDAD'])
plt.title('Boxplot de Edad')
plt.ylabel('Edad')
plt.show()

num_vacios = (df_edt_cleaned['FNACIMIENTO'] == '').sum()
print(f"El número de valores vacíos ('') en la columna 'FNACIMIENTO' es: {num_vacios}")

## SEX

In [ ]:
print("Value counts of sex column before data cleaning:")
print(df_edt_cleaned['SEXO'].value_counts())

# Removing non-valid sex values and encoding values 
df_edt_cleaned = df_edt_cleaned[df_edt_cleaned['SEXO'].isin(['M', 'H'])].copy()
map_sex = {'M': 1, 'H': 0}
df_edt_cleaned.loc[:, 'SEXO'] = df_edt_cleaned['SEXO'].map(map_sex)

print("\nValue counts of sex column after data cleaning:")
print(df_edt_cleaned['SEXO'].value_counts())

print(f"\nAfter excluding by non-valid sex values, the number of rows is: {df_edt_cleaned.shape[0]:,}")

## Fecha demanda

In [ ]:
print(f"\nThe number of nan in FECHADEMANDA is {df_edt_cleaned['FECHADEMANDA'].isnull().sum()}")

df_edt_cleaned['FECHADEMANDA'] = pd.to_datetime(df_edt_cleaned['FECHADEMANDA'], format='%Y%m%d')




# Creation of new variables

In [ ]:
df_edt_new_var = df_edt_cleaned.copy()

print(f"\nThe number of nan in PRIORIDAD is {df_edt_new_var['PRIORIDAD'].isnull().sum()}")

# P1_predicted is the priority predicted during the call, assigned by the system
# 1 is maximum priority, 0 is non-maximum priority
df_edt_new_var['P1_predicted'] = np.where(df_edt_new_var['PRIORIDAD']==1, 1, 0)

# Create the day of the week (number, Monday=0, Sunday=6) variable
df_edt_new_var['day_week'] = df_edt_new_var['FECHADEMANDA'].dt.dayofweek

# Create the month (number) variable
df_edt_new_var['month'] = df_edt_new_var['FECHADEMANDA'].dt.month

# Create the year variable
df_edt_new_var['year'] = df_edt_new_var['FECHADEMANDA'].dt.year

# Display the new columns and the original date column
print(df_edt_new_var[['FECHADEMANDA', 'day_week', 'month', 'year']].head())


## DATA QUALITY

In [ ]:
print(f"\nThe number of nan in IDTIPODEMANDA1 is {df_edt_cleaned['IDTIPODEMANDA1'].isnull().sum()}")

df_edt_new_var.head(50)

In [ ]:
id_tarjeta_deseado = '81SNNM0DHR'  # Asegúrate de que el ID sea una cadena si así está en tu DataFrame

tabla_filtrada = df_edt_new_var[df_edt_new_var['IDTARJETASANITARIA'] == id_tarjeta_deseado]
tabla_filtrada.head(10)

In [ ]:
print(f"El NUHSA {id_tarjeta_deseado} aparece en el dataset {len(tabla_filtrada)} veces")

In [ ]:
print("IDTIPORECURSO", df_edt_cleaned['IDTIPORECURSO'].value_counts())